# Travel Planning Agent

This notebook implements a grounded itinerary-planning workflow for a
fictional trip to AgentsVille. It is not a free-form travel chatbot: the final
answer must satisfy typed data contracts, use supplied source records, and pass
explicit business checks.

The project demonstrates four applied LLM patterns:

1. **Structured contracts** — Pydantic validates the vacation request and the
   nested `TravelPlan` returned by the model.
2. **Chain-of-Thought-style task decomposition** — the initial itinerary prompt
   gives the model an explicit sequence of planning and validation checks.
3. **Evaluation** — Python checks dates, arithmetic, budget, interest coverage,
   and source-record integrity; an LLM evaluator interprets weather suitability.
4. **ReAct revision** — after traveler feedback, the model iteratively requests
   local tools, receives their results, repairs the plan, and submits the exact
   candidate that passed evaluation.

The workflow is organized as follows:

1. configure the API client and model routing;
2. validate the travelers, destination, dates, interests, and budget;
3. retrieve mocked weather and activity data for the complete trip;
4. generate an initial structured itinerary;
5. evaluate the candidate against business constraints;
6. define the tools exposed to the revision agent;
7. revise the itinerary to incorporate feedback requesting at least two
   activities per day; and
8. run an independent final evaluation before presenting the result.

`project_lib.py` contains the Udacity-provided chat wrapper, mocked APIs, and
fictional reference data. This notebook contains the project-specific domain
models, prompts, evaluators, tool interface, and ReAct orchestration. Keeping
this boundary explicit distinguishes supplied runtime scaffolding from the
implementation completed in this project.


## 1. Environment and model routing

The API credential is read from the environment rather than stored in the
notebook. `OPENAI_BASE_URL` defaults to the Udacity-hosted, Vocareum-compatible
endpoint used by the reference environment and can be overridden for another
compatible environment.

The model enum keeps routing decisions explicit: generation and repeated
helper calls can use a cost-conscious model, while quality-sensitive semantic
evaluation can use a stronger model. This separation is useful in real systems
because model quality, latency, and cost are business constraints too.

In [ ]:
# Optional compatibility path for the hosted notebook environment when running in VS Code.
# import os
# import sys
#
# workspace_directory = "/workspace"
# if workspace_directory not in sys.path and os.path.exists(workspace_directory):
#     sys.path.append(workspace_directory)

In [ ]:
# Optional bootstrap for an uninitialized kernel. The locked uv environment
# is the canonical setup.
# %pip install -q json-repair==0.63.3 numexpr==2.14.2 openai==3.3.1 pandas==3.0.5 pydantic==2.13.4 python-dotenv==1.2.3

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in the environment or a local .env file.")

base_url = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
client = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
# Model routing for this project:
# - GPT_56_TERRA: use for quality-sensitive evaluation tasks.
# - GPT_56_LUNA: use for cost-sensitive, high-volume generation tasks.
# - GPT_5_NANO: use for simple, high-frequency helper tasks where minimizing cost matters most.
#
# Luna is the default operational profile. Route quality-sensitive semantic
# evaluation to Terra when measured reliability justifies the added cost.

from enum import Enum


class OpenAIModel(str, Enum):
    GPT_56_TERRA = "gpt-5.6-terra"
    GPT_56_LUNA = "gpt-5.6-luna"
    # GPT_5_NANO = "gpt-5-nano"  # Optional when the endpoint supports this route.


MODEL = OpenAIModel.GPT_56_LUNA

## 2. Vacation request and group semantics

`VacationInfo` is the structured business request. It contains two travelers,
one shared destination, an inclusive trip interval, and one budget for the
group itinerary.

Yuri and Hiro are travel companions following the same daily plan. An activity
therefore appears once in the shared itinerary even when only one companion's
interests motivate the choice. Across the complete trip, however, each traveler
must receive at least one match between an activity's `related_interests` and
their own interests. A single activity that serves both travelers through
different interests is preferred when the other constraints are already met,
but a literal intersection of their interest lists is not required.

In a deployed product, a conversational intake flow could collect these fields
and construct the same validated object before planning begins.

In [ ]:
VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            "interests": ["tennis", "cooking", "comedy", "technology"],
        },
        {
            "name": "Hiro",
            "age": 25,
            "interests": ["reading", "music", "theatre", "art"],
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",
    "date_of_departure": "2025-06-12",
    "budget": 130,
}

In [ ]:
from datetime import date
from pprint import pprint

from pydantic import BaseModel

from project_lib import Interest


class Traveler(BaseModel):
    """One traveler and the interests used for activity matching."""

    name: str
    age: int
    interests: list[Interest]


class VacationInfo(BaseModel):
    """Validated inputs for one group itinerary request."""

    travelers: list[Traveler]
    destination: str
    date_of_arrival: date
    date_of_departure: date
    budget: int


vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)
pprint(vacation_info.model_dump())

vacation_payload = vacation_info.model_dump()
assert set(vacation_payload) == {
    "travelers",
    "destination",
    "date_of_arrival",
    "date_of_departure",
    "budget",
}
assert all(isinstance(traveler, Traveler) for traveler in vacation_info.travelers)
assert isinstance(vacation_info.date_of_arrival, date)
assert isinstance(vacation_info.date_of_departure, date)
assert isinstance(vacation_info.budget, int)
print("VacationInfo validation passed.")

## 3. Grounded weather and activity data

The model should not invent the facts needed to make the plan. The following
cells retrieve the mocked weather forecast and authoritative activity catalog
for every date from arrival through departure, inclusive.

The initial agent receives these records as prompt context. Later, Python
evaluators compare every selected `Activity` object with the mocked API's
authoritative record. The revision agent can also query the catalog one date at
a time through a tool. This mirrors a production pattern in which current
inventory, pricing, policies, or schedules come from business systems rather
than model memory.

The DataFrames are inspection views only; the underlying lists are the context
sent to the itinerary agent.

In [ ]:
import pandas as pd

from project_lib import call_weather_api_mocked


pd.set_option("display.max_colwidth", None)

trip_dates = pd.date_range(
    start=vacation_info.date_of_arrival,
    end=vacation_info.date_of_departure,
    freq="D",
)
weather_for_dates = [
    call_weather_api_mocked(
        date=timestamp.strftime("%Y-%m-%d"),
        city=vacation_info.destination,
    )
    for timestamp in trip_dates
]

weather_for_dates_df = pd.DataFrame(weather_for_dates)
weather_for_dates_df

In [ ]:
from project_lib import call_activities_api_mocked


activities_for_dates = [
    activity
    for timestamp in trip_dates
    for activity in call_activities_api_mocked(
        date=timestamp.strftime("%Y-%m-%d"),
        city=vacation_info.destination,
    )
]

activities_for_dates_df = pd.DataFrame(activities_for_dates)
activities_for_dates_df.head(2)

## 4. Initial itinerary generation

The first stage converts one `VacationInfo` request plus grounded weather and
activity records into a `TravelPlan`.

The Pydantic classes below define the exact nested output contract. The system
prompt then specifies the agent's role, planning task, concise analysis format,
JSON requirements, and factual context. Its step-by-step instructions cover
dates, weather, chronological ordering, non-overlap, individual traveler
interests, budget, pacing, and final validation.

The model returns two labeled sections. `ANALYSIS` is the concise checklist-style
summary required by the notebook's response contract; it is not an exhaustive
transcript of hidden reasoning. `FINAL OUTPUT` contains the JSON instance. Python extracts the JSON
code block and validates it as `TravelPlan`, so plausible prose alone cannot
enter the rest of the workflow.

**Input:** `VacationInfo` user message + weather/activity context  
**Output:** validated initial `TravelPlan` (`travel_plan_1`)

In [ ]:
# Structured output contract for the itinerary-planning workflow.

from datetime import date
from datetime import datetime


class Weather(BaseModel):
    temperature: float
    temperature_unit: str
    condition: str


class Activity(BaseModel):
    activity_id: str
    name: str
    start_time: datetime
    end_time: datetime
    location: str
    description: str
    price: int
    related_interests: list[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity
    reasons_for_recommendation: list[str]


class ItineraryDay(BaseModel):
    date: date
    weather: Weather
    activity_recommendations: list[ActivityRecommendation]


class TravelPlan(BaseModel):
    city: str
    start_date: date
    end_date: date
    total_cost: int
    itinerary_days: list[ItineraryDay]

In [ ]:
import json
from typing import Optional

from project_lib import ChatAgent

ITINERARY_AGENT_SYSTEM_PROMPT = f"""
## Role

You are a careful, helpful, and customer-first itinerary planning agent.

## Task

Create a balanced, complete day-by-day itinerary using the VacationInfo
provided by the user and the Weather Data and Activities Data in the Context.

Plan the itinerary step by step using the following process:

1. Read the destination, arrival date, departure date, budget, travelers,
   and traveler interests from the user's VacationInfo.

2. Create one itinerary day for every date from arrival through departure.

3. Match each itinerary day with the supplied weather data for that date.

4. Select only activities listed in Activities Data and place each selected
   activity on its correct date.

5. Arrange the selected activities for each date in chronological order.
   Ensure that their time ranges do not overlap. One activity may start exactly
   when another activity ends, but it must not start before the previous
   activity ends.

6. Choose activities that collectively reflect the interests of all travelers.

   All travelers attend each selected activity as one group, even when an
   activity was selected primarily because it matches one traveler's interests.

   Across the complete itinerary, ensure that every traveler has at least one
   selected activity for which the activity's `related_interests` field contains
   at least one of that traveler's own interests.

   An individual activity does not need to match every traveler, and travelers
   do not need to share the same interest.

   Prefer activities that match at least one traveler's interests over
   activities that match none of the travelers' interests.

   After satisfying the date, weather, schedule, daily coverage, and budget
   constraints, prefer activities that match the interests of more than one
   traveler, even when each traveler is matched through a different interest.

7. During rainy, stormy, or otherwise incompatible weather, avoid
   outdoor-only activities unless the activity description explicitly
   provides a suitable indoor backup.

8. Include at least one activity per day.

9. Calculate `total_cost` as the sum of the prices of all selected activities
   and ensure it does not exceed the trip budget.

10. Prioritize traveler value. Prefer strong interest fit, weather suitability,
    and reasonable pacing over a larger number of activities or greater
    spending. Do not add activities merely to fill the schedule or use the
    remaining budget.

11. Before answering, verify the trip dates, weather, activity dates,
    schedule conflicts, traveler interest coverage, daily coverage,
    reasonable pacing, and total cost.

Use only the supplied VacationInfo, Weather Data, and Activities Data.
Do not invent activities, weather, prices, dates, interests, or other
factual details.

Copy all fields of every selected activity exactly from Activities Data.

## Output Format

Respond using exactly two labeled sections:

ANALYSIS:
- Date and weather checks: briefly explain how every trip date was covered
  and matched with its weather.
- Schedule checks: confirm that each day's activities are arranged in
  chronological order and that no activity time ranges overlap.
- Activity and interest checks: identify which traveler(s) and individual
  interest(s) are matched by each selected activity. Highlight activities
  whose `related_interests` match more than one traveler. If a selected
  activity matches none of the travelers' interests, briefly explain why
  it was selected.
- Weather compatibility checks: briefly explain why the selected activities
  are suitable for each day's weather.
- Budget check: show the activity-price calculation and compare the total
  with the trip budget.
- Pacing and customer-value checks: briefly confirm that the itinerary is
  reasonably paced and that activities were not added merely to fill the
  schedule or use the remaining budget.
- Final validation: confirm that every trip date has at least one activity,
  every traveler has at least one interest match across the complete
  itinerary, no activity times overlap, and all requirements are satisfied.

Keep the ANALYSIS concise. It should summarize the checks performed rather
than repeat the complete itinerary JSON.

FINAL OUTPUT:
Immediately after the FINAL OUTPUT label, return exactly one Markdown code
block whose opening fence is ```json and whose closing fence is ```.
The code block must contain one complete, valid JSON object conforming to the
TravelPlan JSON Schema below.

Use valid JSON only:
- Use double quotes.
- Do not include comments.
- Do not include trailing commas.
- Do not include any text inside the JSON code block other than the JSON object.
- Output a TravelPlan instance, not the JSON Schema itself.

## TravelPlan JSON Schema

{json.dumps(
    TravelPlan.model_json_schema(),
    indent=2,
    ensure_ascii=False,
)}

## Context

VacationInfo:
{vacation_info.model_dump_json(indent=2)}

Weather Data:
{json.dumps(
    weather_for_dates,
    indent=2,
    ensure_ascii=False,
    default=str,
)}

Activities Data:
{json.dumps(
    activities_for_dates,
    indent=2,
    ensure_ascii=False,
    default=str,
)}
""".strip()


assert "TASK" in ITINERARY_AGENT_SYSTEM_PROMPT.upper()
assert "OUTPUT FORMAT" in ITINERARY_AGENT_SYSTEM_PROMPT.upper()


class ItineraryAgent(ChatAgent):
    """An agent that plans itineraries based on vacation information, weather, and activities."""
    system_prompt = ITINERARY_AGENT_SYSTEM_PROMPT

    def get_itinerary(
        self,
        vacation_info: VacationInfo,
        model: Optional[OpenAIModel] = None,
    ) -> TravelPlan:
        """Generates a travel itinerary based on the provided vacation information."""
        from project_lib import print_in_box
        response = (
            self.chat(
                user_message=vacation_info.model_dump_json(indent=2),
                add_to_messages=False,
                model=model or self.model,
            )
            or ""
        ).strip()

        print_in_box(response, "Raw Response")

        # Isolate the JSON payload from the required two-section response.
        json_text = response.strip()

        if "```json" in json_text:
            json_text = json_text.split("```json")[1].split("```")[0].strip()

        try:
            travel_plan = TravelPlan.model_validate_json(json_text)
            return travel_plan
        except Exception:
            print("Error validating the following text as TravelPlan JSON:")
            print(json_text)
            raise

itinerary_agent = ItineraryAgent(client=client, model=MODEL)

In [ ]:
travel_plan_1 = itinerary_agent.get_itinerary(
    vacation_info=vacation_info,
    model=MODEL,
)
print("Initial itinerary generated and validated as TravelPlan.")

## 5. Evaluation layer

Generation and acceptance are deliberately separate. `get_eval_results()`
runs a list of validators, catches domain-level `AgentError` failures, and
returns a structured `EvaluationResults` object.

Most hard constraints are deterministic:

- arrival/departure dates match the request;
- `total_cost` equals the sum of selected prices;
- every selected activity exactly matches an authoritative mocked-API record;
- every traveler receives at least one interest match; and
- the plan remains within budget.

Weather suitability requires interpreting natural-language descriptions, such
as whether an outdoor event has an indoor rain backup, so that check uses a
small LLM evaluator with a tightly constrained enum output.

The initial plan is evaluated before revision. Later, the same suite becomes a
tool the revision agent can call, and the notebook runs it once more outside
the agent as the final application-side acceptance gate.

In [ ]:
from collections.abc import Callable


class AgentError(Exception):
    """A domain validation failure that the revision workflow can address."""


class EvaluationResults(BaseModel):
    success: bool
    failures: list[str]
    eval_functions: list[str]


EvaluationFunction = Callable[[VacationInfo, TravelPlan], None]


def get_eval_results(
    vacation_info: VacationInfo,
    final_output: TravelPlan,
    eval_functions: list[EvaluationFunction],
) -> EvaluationResults:
    """Run all supplied validators and collect domain failures."""
    from project_lib import print_in_box

    if not isinstance(vacation_info, VacationInfo):
        raise ValueError("vacation_info must be an instance of VacationInfo")
    if not isinstance(final_output, TravelPlan):
        raise ValueError("final_output must be an instance of TravelPlan")
    if not isinstance(eval_functions, list) or not all(
        callable(function) for function in eval_functions
    ):
        raise ValueError("eval_functions must be a list of callable functions")

    failures = []
    for eval_function in eval_functions:
        try:
            eval_function(vacation_info, final_output)
        except AgentError as exc:
            failure = str(exc)
            print_in_box(failure, title="Evaluation Error")
            failures.append(failure)

    return EvaluationResults(
        success=not failures,
        failures=failures,
        eval_functions=[function.__name__ for function in eval_functions],
    )

In [ ]:
# Date-range integrity checks.

def eval_start_end_dates_match(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verify that the plan uses the requested arrival and departure dates.

    Args:
        vacation_info (VacationInfo): The validated trip request.
        final_output (TravelPlan): The itinerary being evaluated.

    Raises:
        AgentError: If either trip boundary differs from the request or the
            plan starts after it ends.
    """
    if (
        vacation_info.date_of_arrival != final_output.start_date
        or vacation_info.date_of_departure != final_output.end_date
    ):
        raise AgentError(
            f"Dates do not match: {vacation_info.date_of_arrival} != {final_output.start_date} or {vacation_info.date_of_departure} != {final_output.end_date}"
        )

    if final_output.start_date > final_output.end_date:
        raise AgentError(
            f"Start date is after end date: {final_output.start_date} > {final_output.end_date}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_start_end_dates_match],
)

In [ ]:
# Cost arithmetic and budget-limit checks.


def eval_total_cost_is_accurate(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verify that the stated total equals the selected activity prices.

    Args:
        vacation_info (VacationInfo): The validated trip request.
        final_output (TravelPlan): The itinerary being evaluated.

    Raises:
        AgentError: If the recomputed activity total differs from `total_cost`.
    """
    actual_total_cost = 0

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            actual_total_cost += activity_recommendation.activity.price

    stated_total_cost = int(final_output.total_cost)

    if actual_total_cost != stated_total_cost:
        raise AgentError(
            f"Stated total cost does not match calculated total cost: {actual_total_cost} != {stated_total_cost}"
        )
    
def eval_total_cost_is_within_budget(vacation_info: VacationInfo, final_output: TravelPlan):
    """Verify that the stated total remains within the requested budget.

    Args:
        vacation_info (VacationInfo): The validated trip request and budget.
        final_output (TravelPlan): The itinerary being evaluated.

    Raises:
        AgentError: If `total_cost` exceeds the request budget.
    """
    stated_total_cost = int(final_output.total_cost)
    if stated_total_cost > vacation_info.budget:
        raise AgentError(
            f"Total cost exceeds budget: {stated_total_cost} > {vacation_info.budget}"
        )

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_total_cost_is_accurate, eval_total_cost_is_within_budget],
)

In [ ]:
# Verify that every selected activity is identical to its authoritative catalog record.

def eval_itinerary_events_match_actual_events(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verify selected activities against authoritative mocked-API records.

    Args:
        vacation_info (VacationInfo): The request associated with the plan.
        final_output (TravelPlan): The itinerary containing selected activities.

    Raises:
        AgentError: If an activity ID is missing or any copied activity field
            differs from the authoritative record.
    """
    from project_lib import call_activity_by_id_api_mocked
    event_ids_not_matching = []
    event_ids_missing = []

    for itinerary_day in final_output.itinerary_days:
        for activity_recommendation in itinerary_day.activity_recommendations:
            event_id = activity_recommendation.activity.activity_id
            reference_event = call_activity_by_id_api_mocked(event_id)

            if reference_event is None:
                event_ids_missing.append(event_id)

            elif Activity(**reference_event) != activity_recommendation.activity:
                print(
                    "---\n"
                    f"Event ID {event_id} does not match the reference event:\n"
                    f"Reference Event: {reference_event}\n"
                    f"Activity Event: {activity_recommendation.activity.model_dump()}"
                )
                event_ids_not_matching.append(event_id)
    if event_ids_missing or event_ids_not_matching:
        raise AgentError(
            f"Event IDs missing: {event_ids_missing}\nEvent IDs not matching: {event_ids_not_matching}"
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_events_match_actual_events],
)

In [ ]:
# Verify complete-trip interest coverage for each traveler.

def eval_itinerary_satisfies_interests(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verify that every traveler receives at least one interest match.

    A match occurs when a selected activity has at least one related interest
    in common with the individual traveler's interests.

    Args:
        vacation_info (VacationInfo): The travelers and their interests.
        final_output (TravelPlan): The itinerary containing selected activities.

    Raises:
        AgentError: If any traveler has no matching selected activity.
    """
    traveler_to_interests = {}
    traveler_to_interest_hit_counts = {}

    for traveler in vacation_info.travelers:
        traveler_to_interests[traveler.name] = traveler.interests
        traveler_to_interest_hit_counts[traveler.name] = 0

    for traveler_name, interests in traveler_to_interests.items():
        for itinerary_day in final_output.itinerary_days:
            for activity_recommendation in itinerary_day.activity_recommendations:
                # Count a match when the activity and traveler share at least one interest.
                matching_interests = set(traveler_to_interests[traveler_name]) & set(
                    activity_recommendation.activity.related_interests
                )

                if matching_interests:
                    traveler_to_interest_hit_counts[traveler_name] += 1

    travelers_with_no_interest_hits = [
        traveler
        for traveler, interest_hit_count in traveler_to_interest_hit_counts.items()
        if interest_hit_count == 0
    ]

    # Reject plans that leave any traveler without an interest match.
    if travelers_with_no_interest_hits:
        raise AgentError(
            f"Travelers {travelers_with_no_interest_hits} have no matching activities in the itinerary."
        )


get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_satisfies_interests],
)

In [ ]:
# Semantic weather-suitability evaluation for activity descriptions.

ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You are an experienced travel-plan evaluator specializing in determining
whether an activity is compatible with a given weather condition.

## Task

Evaluate whether the supplied activity can proceed under the supplied weather
condition.

The user message will contain:

- Activity: the activity name
- Description: the activity description
- Weather Condition: the weather condition for the scheduled day

Determine compatibility using only the supplied activity description and
weather condition.

Follow these rules:

1. If the activity is explicitly held indoors, classify it as compatible.

2. If the weather is clear, sunny, cloudy, or otherwise suitable for outdoor
   activities, classify both indoor and outdoor activities as compatible.

3. If the activity is outdoor-only and the weather includes rain,
   thunderstorms, strong winds, or another unsuitable condition, classify it
   as incompatible.

4. Before classifying an outdoor activity as incompatible, check its
   description for a suitable indoor or weather-safe backup.

5. If a suitable backup is explicitly provided, classify the activity as
   compatible because it can proceed using that backup.

6. If there is not enough information to determine that the weather would
   disrupt the activity, default to compatible.

Do not invent indoor facilities, backup arrangements, or weather details that
are not stated in the input.

## Output Format

Respond using exactly this structure:

REASONING:
- <Provide one brief reason based on the activity description and weather
  condition.>

FINAL ANSWER:
<Output exactly one permitted enum value>

The permitted enum values are:

- IS_COMPATIBLE
- IS_INCOMPATIBLE

Output exactly one enum value after the FINAL ANSWER label.
Do not output a list, square brackets, quotation marks, or additional text
after the enum value.
Do not mention either enum value in the REASONING section.

## Examples

### Example 1: Indoor activity during a thunderstorm

INPUT:
Activity: FutureTech Breakfast Meet-Up
Description: The event is held indoors in the Innovation Atrium.
Weather Condition: thunderstorm

OUTPUT:
REASONING:
- The activity is explicitly held indoors, so the thunderstorm does not
  prevent it from taking place.

FINAL ANSWER:
IS_COMPATIBLE

### Example 2: Outdoor-only activity during a thunderstorm

INPUT:
Activity: Trail Tales: Writing & Hiking Adventure
Description: The activity takes place outdoors and does not mention an indoor
or weather-safe backup.
Weather Condition: thunderstorm

OUTPUT:
REASONING:
- The activity is outdoor-only, and no suitable backup is provided for the
  thunderstorm.

FINAL ANSWER:
IS_INCOMPATIBLE

### Example 3: Outdoor activity with an indoor backup

INPUT:
Activity: Soundtrack Picnic: Lunchtime Movies & Melodies
Description: The event is normally held outdoors but will move indoors to
Harmony Hall in case of rain.
Weather Condition: thunderstorm

OUTPUT:
REASONING:
- The description provides an indoor backup when outdoor weather is unsuitable.

FINAL ANSWER:
IS_COMPATIBLE
""".strip()



def eval_activities_and_weather_are_compatible(
    vacation_info: VacationInfo, final_output: TravelPlan
):
    """Verifies that no outdoor-only activities are scheduled during inclement weather conditions.

    Args:
        vacation_info (VacationInfo): Contains the vacation details
        final_output (TravelPlan): Contains the itinerary details including daily activities and weather conditions

    Raises:
        AgentError: If any outdoor activities are scheduled during weather conditions that could ruin them
    """
    from project_lib import do_chat_completion

    activities_that_are_incompatible = []

    for itinerary_day in final_output.itinerary_days:
        weather_condition = itinerary_day.weather.condition

        for activity_recommendation in itinerary_day.activity_recommendations:
            resp = do_chat_completion(
                messages=[
                    {
                        "role": "system",
                        "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": f"Activity: {activity_recommendation.activity.name}\nDescription: {activity_recommendation.activity.description}\nWeather Condition: {weather_condition}",
                    },
                ],
                client=client,
                # Route frequent binary classification calls to the latency- and
                # cost-efficient evaluator model.
                model=OpenAIModel.GPT_56_LUNA,
            )

            final_section = (resp or "").rsplit("FINAL ANSWER:", maxsplit=1)
            if len(final_section) != 2:
                raise RuntimeError(f"Missing FINAL ANSWER in evaluator response: {resp}")
            verdict = final_section[-1].strip().splitlines()[0].strip()
            if verdict == "IS_COMPATIBLE":
                is_compatible = True
            elif verdict == "IS_INCOMPATIBLE":
                is_compatible = False
            else:
                raise RuntimeError(f"Unexpected compatibility verdict: {verdict}")

            if is_compatible:
                print(
                    f"[PASS] Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are compatible."
                )

            else:
                activities_that_are_incompatible.append(
                    activity_recommendation.activity.name
                )
                print(
                    f"[FAIL] Activity {activity_recommendation.activity.name} (on {itinerary_day.date}) and weather '{weather_condition}' are incompatible."
                )

    if activities_that_are_incompatible:
        raise AgentError(
            f"Activities that may be ruined by inclement weather: {activities_that_are_incompatible}"
        )


eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[
        eval_activities_and_weather_are_compatible
    ],
)

eval_results

In [ ]:
# Consolidated baseline evaluation of the initial itinerary.

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

eval_results.model_dump()

## 6. Tools available to the revision agent

The notebook exposes four Python functions through a text-based tool protocol:

| Tool | Purpose | Key argument |
|---|---|---|
| `calculator_tool` | Calculate exact arithmetic rather than relying on model arithmetic | `input_expression: str` |
| `get_activities_by_date_tool` | Retrieve authoritative activity candidates for one date and city | `date: str`, `city: str` |
| `run_evals_tool` | Evaluate one complete candidate `TravelPlan` and return `success` plus failures | `travel_plan: TravelPlan` |
| `final_answer_tool` | Submit the exact passing plan and signal the loop to stop | `final_output: TravelPlan` |

The LLM does not execute these functions and cannot inspect their Python
bodies. `get_tool_descriptions_string()` converts each function name and
docstring into text for the system prompt. The notebook retains the actual
function objects in `ALL_TOOLS`; its dispatcher later matches the model's
`tool_name` string and invokes the corresponding function with the supplied
JSON arguments.

In [ ]:
def get_tool_descriptions_string(fns: list[Callable]) -> str:
    """Format function names and docstrings for insertion into a prompt."""
    descriptions = []
    for function in fns:
        function_doc = function.__doc__ or "No description provided."
        descriptions.append(f"* `{function.__name__}`: {function_doc}")
    return "\n".join(descriptions)

In [ ]:
# Deterministic arithmetic tool exposed to the revision agent.
def calculator_tool(input_expression: str) -> float:
    """Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    """
    import numexpr as ne

    return float(ne.evaluate(input_expression))


assert calculator_tool("1 + 1") == 2.0

print(get_tool_descriptions_string([calculator_tool]))

In [ ]:
# Authoritative activity-catalog lookup exposed to the revision agent.
def get_activities_by_date_tool(date: str, city: str) -> list[dict]:
    """Retrieves activities available for a specified date and city.

    Use this tool when revising an itinerary and you need to find activities
    available on a particular day.

    Args:
        date (str): The date for which to retrieve activities, formatted as
            YYYY-MM-DD. Supported dates are from 2025-06-10 through
            2025-06-15, inclusive.
        city (str): The city for which to retrieve activities. The only
            supported value is exactly "AgentsVille".

    Returns:
        list[dict]: A list of activity dictionaries available for the specified
            date and city. Returns an empty list if the city is unsupported,
            the date format is invalid, the date is outside the supported
            range, or no activities are available.
    """
    from project_lib import call_activities_api_mocked

    resp = call_activities_api_mocked(date=date, city=city)

    return [Activity.model_validate(activity).model_dump() for activity in resp]



assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0

print(get_tool_descriptions_string([get_activities_by_date_tool]))

In [ ]:
def run_evals_tool(travel_plan: TravelPlan) -> dict:
    """Evaluate one complete candidate travel plan.

    Args:
        travel_plan (TravelPlan): A complete candidate plan conforming to the
            TravelPlan schema.

    Returns:
        dict: A dictionary containing `success` (bool) and `failures`
            (list[str]).
    """
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)

    results = get_eval_results(
        vacation_info=vacation_info,
        final_output=travel_plan,
        eval_functions=ALL_EVAL_FUNCTIONS,
    )
    return {
        "success": results.success,
        "failures": results.failures,
    }


print(get_tool_descriptions_string([run_evals_tool]))

In [ ]:
run_evals_tool(travel_plan=travel_plan_1)

In [ ]:
def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Return the validated final travel plan and end the ReAct loop.

    Args:
        final_output (TravelPlan): The complete final plan to return.

    Returns:
        TravelPlan: The submitted final plan.
    """
    return final_output


print(get_tool_descriptions_string([final_answer_tool]))

In [ ]:
# Tool registry used by prompt construction and the local action dispatcher.
ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))

## 7. ReAct itinerary revision

The business goal of the second agent is to incorporate traveler feedback
while preserving valid parts of the original plan. Passing evaluation is the
completion gate, not a substitute for that revision goal.

This implementation represents ReAct as an explicit text protocol so the
orchestration boundary remains inspectable:

1. Python places the complete system prompt and original `TravelPlan` in the
   chat history.
2. The LLM returns a concise `THOUGHT` describing the next decision and exactly
   one `ACTION` JSON object containing a tool name and arguments.
3. The synchronous model call returns; Python looks for the literal `ACTION:`
   marker, parses the JSON, and executes the selected local function.
4. Python adds the tool result to history as an `OBSERVATION` user message.
5. On the next model call, the LLM sees that observation and chooses one new
   action.
6. A failed evaluation leads to another targeted revision. A successful final
   evaluation must be followed by `final_answer_tool` with the exact same plan.

Python remains in control throughout. `ACTION:` is a parser delimiter, not a
special OpenAI API signal, and the model never directly runs local code.

The protocol probe at the end of the next code cell makes one model call and
checks that the response contains the labels and JSON field expected by the
parser. The following cell then starts a fresh agent instance for the complete
revision run, preventing the probe's history from contaminating the real loop.

In [ ]:
TRAVELER_FEEDBACK = "I want to have at least two activities per day."


def eval_traveler_feedback_is_incorporated(
    vacation_info: VacationInfo,
    final_output: TravelPlan,
) -> None:
    """Check whether the revised plan fully incorporates traveler feedback."""
    feedback_evaluator_prompt = """You are a travel-plan evaluator.

## Task

Determine whether the revised TravelPlan fully incorporates the supplied
Traveler Feedback. Base the decision only on the supplied feedback and plan.

## Output Format

ANALYSIS:
- <One concise explanation of the relevant evidence.>

FINAL OUTPUT:
<Output exactly one of FULLY_INCORPORATED, PARTIALLY_INCORPORATED,
NOT_INCORPORATED, or UNKNOWN.>
REASON: <One concise reason.>
""".strip()

    agent = ChatAgent(
        system_prompt=feedback_evaluator_prompt,
        client=client,
        model=OpenAIModel.GPT_56_TERRA,
    )
    response = agent.chat(
        f"""Traveler Feedback: {TRAVELER_FEEDBACK}
Revised TravelPlan: {final_output.model_dump_json()}"""
    )

    if "FINAL OUTPUT:" not in response:
        raise RuntimeError(
            f"Unexpected evaluator response: {response}. Expected FINAL OUTPUT."
        )

    final_section = response.rsplit("FINAL OUTPUT:", maxsplit=1)[-1].strip()
    verdict = final_section.splitlines()[0].strip()
    if verdict != "FULLY_INCORPORATED":
        raise AgentError(
            "Traveler feedback was not fully incorporated. "
            f"Evaluator response: {final_section}"
        )


ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
    eval_traveler_feedback_is_incorporated,
]

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

In [ ]:
from project_lib import print_in_box


ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
You are a careful and professional itinerary revision agent. You improve an
existing TravelPlan iteratively by using a ReAct loop.

## Task

Revise the original TravelPlan so that it incorporates the Traveler Feedback
while preserving all valid trip requirements and all valid details that do not
need to change.

Use the available tools and their returned observations to diagnose problems,
make the smallest necessary revisions, validate the complete revised plan, and
submit only a plan that passes all evaluations.

## Success Criteria

The task is complete only when:

- The Traveler Feedback has been incorporated.
- The complete candidate TravelPlan satisfies all evaluations.
- The latest run_evals_tool result reports success as True with no failures.
- The TravelPlan submitted to final_answer_tool is exactly the same candidate
  that passed the latest evaluation.

## Available Tools

{get_tool_descriptions_string(ALL_TOOLS)}

## Tool-Use Rules

- Use get_activities_by_date_tool when an activity needs to be added, replaced,
  or refreshed from the authoritative Activities Data. Query the correct date
  and city.

- Select only activities returned by get_activities_by_date_tool. Do not invent
  activities, IDs, names, prices, times, locations, descriptions, or related
  interests.

- The activity-match evaluator compares the complete activity object, not only
  the activity_id. If an evaluation reports an activity ID as not matching, do
  not automatically assume that the ID is invalid. One or more fields of the
  activity object may differ from the authoritative record.

- For every activity reported as not matching, call
  get_activities_by_date_tool for that activity's date, find the returned
  activity with the same activity_id, and replace the complete activity object
  with that authoritative record before changing the activity selection.

- Copy all text, numeric, and list fields from the authoritative activity
  record exactly. Do not shorten, paraphrase, summarize, or reconstruct the
  description. Serialize start_time and end_time as equivalent ISO-8601 JSON
  strings without changing their values.

- reasons_for_recommendation is separate from the authoritative activity
  fields. Every activity recommendation must contain a non-empty
  reasons_for_recommendation list.

- Ensure that activities scheduled on the same day do not overlap in time.

- Maintain one current candidate TravelPlan throughout the loop. When responding
  to an evaluation failure, change only the fields needed to address that
  failure and preserve all other valid fields.

- After adding, removing, or replacing any activity, call calculator_tool before
  calling run_evals_tool again. Sum the prices of every selected activity and
  set total_cost to exactly the value returned by calculator_tool.

- Also use calculator_tool when an evaluation reports that total_cost does not
  match the sum of the selected activity prices.

- Before calling run_evals_tool, verify that every activity recommendation
  contains a complete activity object and a non-empty
  reasons_for_recommendation list.

- When calling run_evals_tool, the travel_plan argument must contain one
  complete TravelPlan JSON object. Do not pass a Python variable name, a Python
  or Pydantic object representation, a partial plan, a JSON Schema, or a list
  of changes.

- When calling final_answer_tool, the final_output argument must contain the
  exact complete TravelPlan JSON object that passed the latest evaluation.

- Distinguish an evaluation failure from a tool-execution error. An evaluation
  result contains success and failures. If an OBSERVATION instead reports
  "Error occurred while calling tool", do not modify the candidate based on
  that error. Retry the same tool call once with the same arguments.

- Any change made after a candidate has been evaluated invalidates that
  evaluation. Evaluate the changed candidate again before submitting it.

## Required ReAct Process

Use a THINK-ACT-OBSERVE cycle. In this notebook, THINK is represented by the
THOUGHT label and ACT is represented by the ACTION label.

1. Your first ACTION must call run_evals_tool on the complete original
   TravelPlan supplied in the user message. Do not revise the original plan
   before this baseline evaluation.

2. In each THOUGHT, briefly summarize the current issue or the relevant result
   from the latest OBSERVATION and explain why the selected tool is the
   appropriate next action.

3. In each ACTION, call exactly one available tool using the required JSON
   format.

4. The Python program will execute the ACTION and add the result to the
   conversation as an OBSERVATION. Do not generate an OBSERVATION yourself.

5. After each OBSERVATION, use its result to choose the single best next action.

6. If run_evals_tool returns success as False with failures, address only the
   reported failures, preserve the remaining valid details, and evaluate the
   complete updated candidate again.

7. Immediately before completion, call run_evals_tool on the exact complete
   candidate that you intend to submit. The initial baseline evaluation does
   not replace this final evaluation.

8. Call final_answer_tool only on the turn immediately following an
   OBSERVATION in which run_evals_tool reports success as True with no failures.

9. Pass final_answer_tool the exact same complete candidate that passed that
   latest evaluation. Calling final_answer_tool ends the ReAct loop.

## Output Format

Every response must contain exactly one THOUGHT and one ACTION in the following
format.

Replace the angle-bracketed placeholders with actual content. Never output the
placeholders literally.

THOUGHT:
<One or two concise sentences describing the current issue or result and why
the selected tool is the appropriate next action. Do not provide exhaustive
step-by-step reasoning.>

ACTION:
{{"tool_name": "<exact available tool name>", "arguments": {{"<exact parameter name>": "<appropriate JSON value>"}}}}

Output requirements:

- Use the exact uppercase labels THOUGHT: and ACTION:.
- Produce exactly one THOUGHT and exactly one ACTION per response.
- After ACTION:, output exactly one valid JSON object.
- The JSON object must contain the top-level keys "tool_name" and "arguments".
- The "tool_name" value must exactly match one available tool name.
- The "arguments" value must be a JSON object containing only the exact
  parameter names required by the selected tool.
- Embed a complete TravelPlan as a JSON object, not as a quoted string, when
  calling run_evals_tool or final_answer_tool.
- Use double quotes for all JSON keys and string values.
- Do not use Python constructors, Python variable names, comments, trailing
  commas, Markdown code fences, or a list of tool calls.
- Do not output an OBSERVATION section.
- Do not output any text after the ACTION JSON object.

## Context

Traveler Feedback:

{TRAVELER_FEEDBACK}

Vacation Information:

{vacation_info.model_dump_json(indent=2)}

The complete original TravelPlan to revise will be supplied in the user message
that starts the ReAct loop.

TravelPlan JSON Schema:

{json.dumps(
    TravelPlan.model_json_schema(),
    indent=2,
    ensure_ascii=False,
)}
""".strip()


class ItineraryRevisionAgent(ChatAgent):
    system_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT
    tools = ALL_TOOLS

    def get_observation_string(self, tool_call_obj) -> str:
        """Execute a parsed tool call and format its result as an OBSERVATION."""

        if "tool_name" not in tool_call_obj:
            return "OBSERVATION: No tool name specified."

        if "arguments" not in tool_call_obj:
            return "OBSERVATION: No arguments specified."

        if not isinstance(tool_call_obj["arguments"], dict):
            return f"OBSERVATION: Arguments should be a dictionary, got {type(tool_call_obj['arguments'])} instead."

        if not isinstance(tool_call_obj["tool_name"], str):
            return f"OBSERVATION: Tool name should be a string, got {type(tool_call_obj['tool_name'])} instead."

        tool_name = tool_call_obj["tool_name"]
        arguments = tool_call_obj["arguments"]

        tool_fn = None

        for tool in self.tools:
            if tool.__name__ == tool_name:
                tool_fn = tool
                break

        if tool_fn is None:
            return f"OBSERVATION: Unknown tool name '{tool_name}' in action string."

        try:
            tool_response = tool_fn(**arguments)
            return f"OBSERVATION: Tool {tool_name} called successfully with response: {tool_response}"
        except Exception as e:
            return f"OBSERVATION: Error occurred while calling tool {tool_name}: {e}"

    def run_react_cycle(
        self,
        original_travel_plan: TravelPlan,
        max_steps: int = 10,
        model: Optional[OpenAIModel] = None,
        client=None,
    ) -> TravelPlan:
        """Runs the ReAct cycle to revise the itinerary based on the evaluation results."""
        from json_repair import repair_json

        # Seed the conversation with the authoritative baseline plan.
        self.add_message(
            role="user",
            content=f"Here is the itinerary for revision:\n{original_travel_plan.model_dump_json()}",
        )
        resp = None

        # Bound the loop to prevent unbounded model and tool calls.
        for _ in range(max_steps):
            # Request one THOUGHT/ACTION response from the model.
            resp = self.get_response(model=model, client=client) or ""

            # Recover from a missing ACTION without executing a tool.
            if "ACTION:" not in resp:
                self.add_message(role="user", content="No action found in response.")
                continue

            action_string = resp.split("ACTION:", maxsplit=1)[1].strip()

            # Parse the JSON payload following the ACTION delimiter.
            try:
                # Repair common structural JSON defects before strict parsing.
                action_string = repair_json(action_string)
                tool_call_obj = json.loads(action_string)
            except json.JSONDecodeError:
                print(f"Invalid JSON in action string: {action_string}")
                self.add_message(
                    role="user",
                    content=f"Invalid JSON in action string: {action_string}",
                )
                continue

            tool_name = tool_call_obj.get("tool_name")

            # Treat final_answer_tool as the terminal action.
            if tool_name == "final_answer_tool":
                try:
                    new_travel_plan = TravelPlan.model_validate(
                        tool_call_obj["arguments"].get(
                            "final_output", tool_call_obj["arguments"]
                        )
                    )
                    return new_travel_plan
                except Exception as e:
                    self.add_message(
                        role="user", content=f"Error validating final answer: {e}"
                    )
                    continue

            else:
                observation_string = self.get_observation_string(
                    tool_call_obj=tool_call_obj
                )
                self.add_message(role="user", content=observation_string)

        raise RuntimeError(
            f"ReAct cycle did not complete within {max_steps} steps. Last response: {resp}"
        )


# Protocol probe: verify the model follows the parser contract.
itinerary_revision_agent = ItineraryRevisionAgent()
protocol_probe = itinerary_revision_agent.chat(
    user_message=(
        "Here is the itinerary for revision:\n"
        f"{travel_plan_1.model_dump_json(indent=2)}"
    ),
    add_to_messages=False,
    model=MODEL,
    client=client,
) or ""

print_in_box(protocol_probe, "ReAct Protocol Probe")
required_protocol_markers = ("THOUGHT:", "ACTION:", '"tool_name"')
missing_protocol_markers = [
    marker for marker in required_protocol_markers if marker not in protocol_probe
]
assert not missing_protocol_markers, (
    "The revision-agent response is missing required protocol markers: "
    f"{missing_protocol_markers}"
)
print("ReAct response contract detected.")

In [ ]:
# ReAct trajectories are stochastic; increase diagnostics or refine the prompt
# if the loop reaches the configured step limit.
itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1,
    max_steps=15,
    model=MODEL,
    client=client,
)
print("Revised itinerary generated and validated as TravelPlan.")

In [ ]:
# Independent application-side acceptance gate.
eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

assert eval_results_2.success, (
    "The revised itinerary did not pass every evaluation. "
    f"Failures: {eval_results_2.failures}"
)
print("All final evaluation functions passed.")
eval_results_2

In [ ]:
from IPython.display import display


for itinerary_day in sorted(travel_plan_2.itinerary_days, key=lambda day: day.date):
    print(f"Date: {itinerary_day.date}")
    print(
        "Weather: "
        f"{itinerary_day.weather.condition} "
        f"({itinerary_day.weather.temperature}°"
        f"{itinerary_day.weather.temperature_unit})"
    )

    ordered_recommendations = sorted(
        itinerary_day.activity_recommendations,
        key=lambda recommendation: recommendation.activity.start_time,
    )
    activities_df = pd.DataFrame(
        [
            recommendation.activity.model_dump()
            for recommendation in ordered_recommendations
        ]
    )
    display(activities_df)

## 8. Optional traveler-facing narrative

After the structured plan passes the final acceptance gate, the validated data
can support a more conversational presentation. This optional step generates a
narrative from the accepted itinerary; it does not participate in planning,
tool selection, or evaluation.

In [ ]:
from project_lib import narrate_my_trip


narrate_my_trip(
    vacation_info=vacation_info,
    itinerary=travel_plan_2,
    client=client,
    model=MODEL,
)

## Reproducibility and handoff

At this point the workflow has produced a structured itinerary and
independently checked it with the complete evaluation suite. The repository
notebook retains every source cell while removing stored model responses,
prompt dumps, execution counts, and ReAct traces. Running the notebook in a
configured environment regenerates fresh outputs; trajectories may vary
because LLM calls are stochastic.

The successful output-bearing notebook submitted to Udacity is preserved in a
separate local `Archive.zip` and excluded from version control. Separating the
execution evidence from the source-clean portfolio artifact keeps the reviewed
run recoverable without making large, stochastic traces part of the code
review surface.

`README.md` provides the broader business interpretation, architecture, LLM
and Python responsibility boundary, CoT versus ReAct comparison, modern
production alternatives, setup instructions, validation evidence, limitations,
and rubric map.
